# 第三部分：数据清洗

处理下载后的原始表：先做单表清洗，再生成宽表、长表和合并表。除 CSV 外，也用 Parquet 进行存储。


In [1]:

from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != 'dshw-p01':
    PROJECT_ROOT = PROJECT_ROOT / 'dshw-p01'
os.makedirs(PROJECT_ROOT, exist_ok=True)

for path in [
    'data/stock', 'data/index', 'data/macro', 'data/finance',
    'data/clean', 'data/combined', 'output'
]:
    os.makedirs(PROJECT_ROOT / path, exist_ok=True)

PROJECT_ROOT


PosixPath('/Users/wonderlab/Desktop/欣欣作业/ds2026/dshw-p01')

In [2]:

import os, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

STOCK_INFO = pd.DataFrame([
    {'code': '000625', 'name': '长安汽车', 'industry': '汽车'},
    {'code': '601127', 'name': '赛力斯', 'industry': '汽车'},
    {'code': '600246', 'name': '万通发展', 'industry': '房地产'},
    {'code': '000560', 'name': '我爱我家', 'industry': '房地产'},
    {'code': '600519', 'name': '贵州茅台', 'industry': '白酒'},
    {'code': '000799', 'name': '酒鬼酒', 'industry': '白酒'},
    {'code': '600578', 'name': '京能电力', 'industry': '能源'},
    {'code': '601016', 'name': '节能风电', 'industry': '能源'},
    {'code': '000063', 'name': '中兴通讯', 'industry': '通讯'},
    {'code': '300136', 'name': '信维通信', 'industry': '通讯'},
])

## 3.1 单表清洗

对 10 只股票原始行情表逐项处理。清洗过程中先把日期列转为 `datetime64` 并设为索引，便于按时间排序和检查；保存清洗结果前再把日期恢复为普通列，方便后续合并。


In [3]:

stock_frames = []
missing_rows = []
dtype_rows = []
clean_log = []
extreme_rows = []
date_index_rows = []

numeric_cols = ['open', 'close', 'high', 'low', 'volume', 'amount']

for path in sorted((PROJECT_ROOT / 'data/stock').glob('stock_*.csv')):
    df = pd.read_csv(path, dtype={'code': str})
    code = df['code'].iloc[0]
    before_rows = len(df)

    for col in df.columns:
        miss = df[col].isna().sum()
        missing_rows.append({
            'file': path.name,
            'code': code,
            'column': col,
            'missing_count': int(miss),
            'missing_ratio': miss / max(len(df), 1)
        })

    before_dtypes = df[['date'] + numeric_cols].dtypes.astype(str).to_dict()
    df['date'] = pd.to_datetime(df['date'])
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    after_dtypes = df[['date'] + numeric_cols].dtypes.astype(str).to_dict()
    for col in ['date'] + numeric_cols:
        dtype_rows.append({
            'file': path.name,
            'code': code,
            'column': col,
            'before_dtype': before_dtypes[col],
            'after_dtype': after_dtypes[col],
            'converted': before_dtypes[col] != after_dtypes[col]
        })

    df = df.set_index('date').sort_index()
    date_index_rows.append({
        'file': path.name,
        'code': code,
        'index_name': df.index.name,
        'index_dtype': str(df.index.dtype),
        'is_monotonic_increasing': bool(df.index.is_monotonic_increasing)
    })

    dup_count = df.reset_index().duplicated(['code', 'date']).sum()
    missing_numeric_before = df[numeric_cols].isna().sum().sum()
    df = df.reset_index().drop_duplicates(['code', 'date'], keep='first')
    df = df.set_index('date').sort_index()
    df[numeric_cols] = df.groupby('code')[numeric_cols].ffill()
    missing_numeric_after = df[numeric_cols].isna().sum().sum()

    df['pct_chg'] = df.groupby('code')['close'].pct_change() * 100
    df['return'] = np.log(df['close'] / df.groupby('code')['close'].shift(1))
    df['is_extreme'] = df['return'].abs() > 0.20

    extreme_sample = df[df['is_extreme']].reset_index()[['date', 'code', 'name', 'industry', 'close', 'return', 'pct_chg']]
    extreme_rows.append(extreme_sample)

    stock_frames.append(df.reset_index())
    clean_log.append({
        'code': code,
        'before_rows': before_rows,
        'after_rows': len(df),
        'duplicates_removed': int(dup_count),
        'numeric_missing_before_ffill': int(missing_numeric_before),
        'numeric_missing_after_ffill': int(missing_numeric_after),
        'extreme_rows': int(df['is_extreme'].sum())
    })

missing_summary = pd.DataFrame(missing_rows)
dtype_log = pd.DataFrame(dtype_rows)
date_index_log = pd.DataFrame(date_index_rows)
clean_log = pd.DataFrame(clean_log)
extreme_detail = pd.concat(extreme_rows, ignore_index=True) if extreme_rows else pd.DataFrame(
    columns=['date', 'code', 'name', 'industry', 'close', 'return', 'pct_chg']
)

stock_clean = pd.concat(stock_frames, ignore_index=True)
stock_clean = stock_clean.merge(STOCK_INFO, on=['code', 'name', 'industry'], how='left')
stock_clean.to_csv(PROJECT_ROOT / 'data/clean/stock_clean.csv', index=False, encoding='utf-8-sig')
stock_clean.to_parquet(PROJECT_ROOT / 'data/clean/stock_clean.parquet', index=False)

clean_log


,code,before_rows,after_rows,duplicates_removed,numeric_missing_before_ffill,numeric_missing_after_ffill,extreme_rows
0,000063,1539,1539,0,0,0,0
1,000560,1540,1540,0,0,0,0
2,000625,1540,1540,0,0,0,0
3,000799,1540,1540,0,0,0,0
4,300136,1540,1540,0,0,0,1
5,600246,1539,1539,0,0,0,0
6,600519,1540,1540,0,0,0,0
7,600578,1540,1540,0,0,0,0
8,601016,1534,1534,0,0,0,0
9,601127,1539,1539,0,0,0,0


### 3.1.1 缺失值检测

先统计每列缺失值的数量和比例。

缺失值可能来自停牌、数据源字段为空，或者某些交易日没有对应的成交金额等字段。由于股票数据本身只包含交易日，节假日通常不会表现为表内缺失，而是日期序列中没有这些日期。


In [4]:
missing_summary.sort_values(['code', 'missing_count', 'column'], ascending=[True, False, True]).head(30)


,file,code,column,missing_count,missing_ratio
9,stock_000063.csv,000063,amount,0,0.0
5,stock_000063.csv,000063,close,0,0.0
1,stock_000063.csv,000063,code,0,0.0
0,stock_000063.csv,000063,date,0,0.0
6,stock_000063.csv,000063,high,0,0.0
3,stock_000063.csv,000063,industry,0,0.0
7,stock_000063.csv,000063,low,0,0.0
2,stock_000063.csv,000063,name,0,0.0
4,stock_000063.csv,000063,open,0,0.0
8,stock_000063.csv,000063,volume,0,0.0


### 3.1.2 缺失值处理

价格和成交量字段使用同一股票内部的向前填充 `ffill`。这样做的原因是要计算连续价格序列和收益率，少量字段缺失时沿用最近一个有效交易日的信息，比直接删除空值整行更能保留样本长度。


In [5]:
clean_log[['code', 'before_rows', 'after_rows', 'numeric_missing_before_ffill', 'numeric_missing_after_ffill']]


,code,before_rows,after_rows,numeric_missing_before_ffill,numeric_missing_after_ffill
0,000063,1539,1539,0,0
1,000560,1540,1540,0,0
2,000625,1540,1540,0,0
3,000799,1540,1540,0,0
4,300136,1540,1540,0,0
5,600246,1539,1539,0,0
6,600519,1540,1540,0,0
7,600578,1540,1540,0,0
8,601016,1534,1534,0,0
9,601127,1539,1539,0,0


### 3.1.3 日期格式统一

所有股票表的 `date` 都被转换为 `datetime64`，并在单表清洗阶段设为索引。这样可以保证按交易日排序、去重和收益率计算都基于统一的时间类型。

In [6]:
date_index_log


,file,code,index_name,index_dtype,is_monotonic_increasing
0,stock_000063.csv,000063,date,datetime64[ns],True
1,stock_000560.csv,000560,date,datetime64[ns],True
2,stock_000625.csv,000625,date,datetime64[ns],True
3,stock_000799.csv,000799,date,datetime64[ns],True
4,stock_300136.csv,300136,date,datetime64[ns],True
5,stock_600246.csv,600246,date,datetime64[ns],True
6,stock_600519.csv,600519,date,datetime64[ns],True
7,stock_600578.csv,600578,date,datetime64[ns],True
8,stock_601016.csv,601016,date,datetime64[ns],True
9,stock_601127.csv,601127,date,datetime64[ns],True


### 3.1.4 数据类型检查

价格、成交量和成交额字段都需要是数值型。下面的表记录转换前后的类型，`converted=True` 表示该字段在清洗时发生了类型转换。


In [7]:

dtype_log.sort_values(['code', 'column']).head(80)


,file,code,column,before_dtype,after_dtype,converted
6,stock_000063.csv,000063,amount,float64,float64,False
2,stock_000063.csv,000063,close,float64,float64,False
0,stock_000063.csv,000063,date,object,datetime64[ns],True
3,stock_000063.csv,000063,high,float64,float64,False
4,stock_000063.csv,000063,low,float64,float64,False
...,...,...,...,...,...,...
63,stock_601127.csv,601127,date,object,datetime64[ns],True
66,stock_601127.csv,601127,high,float64,float64,False
67,stock_601127.csv,601127,low,float64,float64,False
64,stock_601127.csv,601127,open,float64,float64,False


### 3.1.5 重复值处理

重复值按 `code-date` 检测，同一只股票同一天只保留第一条记录。这样能避免同一交易日被重复计入收益率和后续统计。

如果 `duplicates_removed` 为 0，说明原始股票行情表没有发现重复交易日；如果大于 0，则表示已删除对应数量的重复记录。


In [8]:
clean_log[['code', 'before_rows', 'after_rows', 'duplicates_removed']]


,code,before_rows,after_rows,duplicates_removed
0,000063,1539,1539,0
1,000560,1540,1540,0
2,000625,1540,1540,0
3,000799,1540,1540,0
4,300136,1540,1540,0
5,600246,1539,1539,0
6,600519,1540,1540,0
7,600578,1540,1540,0
8,601016,1534,1534,0
9,601127,1539,1539,0


### 3.1.6 离群值标注

这里计算日对数收益率，并把单日收益率绝对值超过 20% 的记录标为 `is_extreme=True`。这些记录只标注、不删除，因为它们可能来自真实的大幅涨跌、复权价格调整、停复牌后的价格跳跃，或者数据源异常。


In [9]:

extreme_summary = clean_log[['code', 'extreme_rows']].sort_values('extreme_rows', ascending=False)
display(extreme_summary)
extreme_detail.sort_values('return', key=lambda s: s.abs(), ascending=False).head(20)


,code,extreme_rows
4,300136,1
0,000063,0
1,000560,0
2,000625,0
3,000799,0
5,600246,0
6,600519,0
7,600578,0
8,601016,0
9,601127,0


,date,code,name,industry,close,return,pct_chg
0,2025-04-07,300136,信维通信,通讯,243.18,-0.223028,-19.990788


## 3.2 宽表与长表转换

把 10 只股票的收盘价先合并为宽表，再用 `pd.melt` 转回长表。
对比：
- 宽表以日期为索引、每列一只股票，适合做相关系数矩阵、组合收益和多资产横向比较。
- 长表字段为 `date, code, close`，更适合分组统计、分面绘图，以及和行业、宏观等其他表继续合并。


In [10]:

close_wide = stock_clean.pivot(index='date', columns='code', values='close').sort_index()
close_long = close_wide.reset_index().melt(id_vars='date', var_name='code', value_name='close')
close_wide.to_csv(PROJECT_ROOT / 'data/clean/stock_close_wide.csv', encoding='utf-8-sig')
close_long.to_csv(PROJECT_ROOT / 'data/clean/stock_close_long.csv', index=False, encoding='utf-8-sig')

print(f'宽表形状: {close_wide.shape}')
print(f'长表形状: {close_long.shape}')
display(close_wide.tail())
close_long.head()


宽表形状: (1540, 10)
长表形状: (15400, 3)


code,000063,000560,000625,000799,300136,600246,600519,600578,601016,601127
date,,,,,,,,,,
2026-05-11,720.32,17.70,144.32,98.22,1536.64,160.73,11812.46,32.39,13.75,93.78
2026-05-12,701.53,17.64,142.30,95.42,1453.27,163.35,11753.63,32.89,13.75,91.21
2026-05-13,715.16,16.90,141.06,95.25,1511.77,163.84,11662.86,36.20,15.12,91.96
2026-05-14,691.77,16.48,138.11,96.55,1362.99,156.11,11646.20,39.83,13.70,89.42
2026-05-15,676.12,16.11,137.65,92.32,1324.43,165.96,11566.20,43.83,13.51,87.54


,date,code,close
0,2020-01-02,000063,602.59
1,2020-01-03,000063,621.29
2,2020-01-06,000063,623.84
3,2020-01-07,000063,622.14
4,2020-01-08,000063,607.01


## 3.3 多表合并

把个股、指数、宏观、财务数据按日期或月份对齐。先准备各表，再执行合并。

### 3.3.1 数据准备：指数、宏观与财务清洗

指数是日度数据，需要统一日期格式并计算沪深 300 等指数收益率。宏观指标是月度数据，因此先把日期转成月份，再整理成长表和宽表。财务指标保留公司-年度-指标的长格式，同时生成一份宽表，方便后面画 ROE。


In [11]:

index_frames = []
for path in sorted((PROJECT_ROOT / 'data/index').glob('index_*.csv')):
    df = pd.read_csv(path, dtype={'code': str})
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'close', 'high', 'low', 'volume', 'amount']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.sort_values(['code', 'date'])
    df['pct_chg'] = df.groupby('code')['close'].pct_change() * 100
    df['index_return'] = np.log(df['close'] / df.groupby('code')['close'].shift(1))
    index_frames.append(df)
index_clean = pd.concat(index_frames, ignore_index=True)
index_clean.to_csv(PROJECT_ROOT / 'data/clean/index_clean.csv', index=False, encoding='utf-8-sig')

macro_frames = []
for path in sorted((PROJECT_ROOT / 'data/macro').glob('macro_*.csv')):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.to_period('M').astype(str)
    macro_frames.append(df[['month', 'indicator', 'value']])
macro_long = pd.concat(macro_frames, ignore_index=True)
macro_wide = macro_long.pivot_table(index='month', columns='indicator', values='value', aggfunc='last').reset_index()
macro_long.to_csv(PROJECT_ROOT / 'data/clean/macro_clean.csv', index=False, encoding='utf-8-sig')

finance = pd.read_csv(PROJECT_ROOT / 'data/finance/finance_ratios.csv', dtype={'code': str})
finance['year'] = pd.to_numeric(finance['year'], errors='coerce').astype('Int64')
finance['value'] = pd.to_numeric(finance['value'], errors='coerce')
finance = finance.dropna(subset=['code', 'year', 'indicator', 'value'])
finance.to_csv(PROJECT_ROOT / 'data/clean/finance_clean.csv', index=False, encoding='utf-8-sig')
finance_wide = finance.pivot_table(index=['code', 'name', 'industry', 'year'], columns='indicator', values='value', aggfunc='first').reset_index()
finance_wide.to_csv(PROJECT_ROOT / 'data/clean/finance_wide.csv', index=False, encoding='utf-8-sig')

index_clean.head(), macro_wide.head(), finance_wide.head()


(        date    code  name      open     close      high       low  \
 0 2020-01-02  000001  上证综指  3066.336  3085.198  3098.100  3066.336   
 1 2020-01-03  000001  上证综指  3089.022  3083.786  3093.819  3074.518   
 2 2020-01-06  000001  上证综指  3070.909  3083.408  3107.203  3065.309   
 3 2020-01-07  000001  上证综指  3085.488  3104.802  3105.451  3084.329   
 4 2020-01-08  000001  上证综指  3094.239  3066.893  3094.239  3059.131   
 
         volume  amount   pct_chg  index_return  
 0  29247020800     NaN       NaN           NaN  
 1  26149666700     NaN -0.045767     -0.000458  
 2  31257584200     NaN -0.012258     -0.000123  
 3  27658311100     NaN  0.693843      0.006914  
 4  29787255300     NaN -1.220980     -0.012285  ,
 indicator    month  cpi_yoy  m2_yoy
 0          2020-01      4.5     8.4
 1          2020-02      5.4     8.8
 2          2020-03      5.2    10.1
 3          2020-04      4.3    11.1
 4          2020-05      3.3    11.1,
 indicator    code  name industry  year  net_pro

### 3.3.2 合并执行

先把个股日度数据与沪深 300 日度收益率按 `date` 左连接，再处理日度股票数据和月度宏观数据的频率差异。这里将每个股票交易日映射到所在月份，再把 CPI 和 M2 接到对应月份上。


In [12]:

merge_log = []
hs300 = index_clean[index_clean['code'] == '000300'][['date', 'close', 'index_return']].rename(
    columns={'close': 'hs300_close', 'index_return': 'hs300_return'}
)
before = len(stock_clean)
combined = stock_clean.merge(hs300, on='date', how='left')
merge_log.append({
    'step': 'stock_left_join_hs300',
    'join_key': 'date',
    'before_rows': before,
    'after_rows': len(combined),
    'row_change': len(combined) - before,
    'explanation': '左连接保留全部个股交易日；行数不变说明沪深300日期键没有造成重复匹配。'
})

before = len(combined)
combined['month'] = combined['date'].dt.to_period('M').astype(str)
combined = combined.merge(macro_wide, on='month', how='left')
merge_log.append({
    'step': 'join_monthly_macro',
    'join_key': 'month',
    'before_rows': before,
    'after_rows': len(combined),
    'row_change': len(combined) - before,
    'explanation': '把月度宏观变量映射到每个交易日；宏观宽表每月一行，因此左连接后行数应保持不变。'
})

combined.to_csv(PROJECT_ROOT / 'data/combined/combined_data.csv', index=False, encoding='utf-8-sig')
combined.to_parquet(PROJECT_ROOT / 'data/combined/combined_data.parquet', index=False)
pd.DataFrame(merge_log)


,step,join_key,before_rows,after_rows,row_change,explanation
0,stock_left_join_hs300,date,15391,15391,0,左连接保留全部个股交易日；行数不变说明沪深300日期键没有造成重复匹配。
1,join_monthly_macro,month,15391,15391,0,把月度宏观变量映射到每个交易日；宏观宽表每月一行，因此左连接后行数应保持不变。


# 第二部分：数据存储与管理（进阶演示）

在已完成 CSV 存储（方式 A）的基础上，额外演示方式 B（Parquet）。

## 2.2 数据存储格式（方式 B：Parquet）

选择方式 B：Parquet。下面演示三件事：只读取需要的列、查看 Schema，以及比较 CSV 与 Parquet 的读取耗时和文件大小。


In [13]:

df_small = pd.read_parquet(PROJECT_ROOT / 'data/clean/stock_clean.parquet', columns=['date', 'code', 'close'])
schema = pq.read_schema(PROJECT_ROOT / 'data/clean/stock_clean.parquet')
print(schema)

t0 = time.time()
pd.read_csv(PROJECT_ROOT / 'data/clean/stock_clean.csv')
csv_time = time.time() - t0
csv_size = os.path.getsize(PROJECT_ROOT / 'data/clean/stock_clean.csv') / 1024

t0 = time.time()
pd.read_parquet(PROJECT_ROOT / 'data/clean/stock_clean.parquet')
pq_time = time.time() - t0
pq_size = os.path.getsize(PROJECT_ROOT / 'data/clean/stock_clean.parquet') / 1024

storage_compare = pd.DataFrame([
    {'format': 'CSV', 'read_seconds': csv_time, 'size_kb': csv_size},
    {'format': 'Parquet', 'read_seconds': pq_time, 'size_kb': pq_size},
])
storage_compare


date: timestamp[ns]
code: string
name: string
industry: string
open: double
close: double
high: double
low: double
volume: double
amount: double
pct_chg: double
return: double
is_extreme: bool
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1529


,format,read_seconds,size_kb
0,CSV,0.009124,1999.310547
1,Parquet,0.001747,831.941406


从本次数据规模看，CSV 和 Parquet 的读取速度差异不会特别稳定，因为文件都不大，系统缓存和单次运行状态会影响耗时。Parquet 的优势主要体现在列式读取和类型保留：当股票数量更多、时间跨度更长，或只需要读取少数列做分析时，Parquet 通常会更省空间，也更适合反复读取。
